In [ ]:
!pip -q install wilds tabicl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 4.0 MB/s eta 0:00:00


=====================================================================
Waterbirds  —  Kernel-LOUIS vs a DFR-style group oracle
---------------------------------------------------------------------
Framing: DFR (Kirichenko et al. 2022) shows frozen features already
contain the core signal, and that RETRAINING THE LAST LAYER on
group-balanced data recovers worst-group accuracy. Here TabICL *is*
the last layer, and our LOO difficulty score is the GROUP-LABEL-FREE
replacement for DFR's group-balanced reweighting. So:

   baseline      = TabICL on the natural (spurious) context
   ours          = TabICL on a LOO-reweighted context      (no group labels)
   DFR-oracle    = TabICL on a group-balanced context      (uses group labels)
   random        = TabICL on a randomly reweighted context (control)

THE DECISIVE DIAGNOSTIC IS REPORTED FIRST: the AUC of the LOO score for minority-group membership. If LOO does not rank minority-group points
above majority ones, no downstream reweighting can work, and every other
number in this script is moot. Do not skip past it to the accuracy table. -->

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [2]:
import torch, torchvision
from torch.utils.data import DataLoader
from wilds import get_dataset

ModuleNotFoundError: No module named 'wilds'

In [ ]:
SEED = 0
BACKEND = "tabicl"      # "tabicl" | "stub" (stub = fast sklearn stand-in)
DATA = "wilds"          # "wilds"  | "synthetic" (synthetic = Waterbirds-shaped sanity data)
N_PCA = 128             # 2048-d ResNet features -> PCA (column attention over 2048 is slow)
N_CONTEXT = 4000        # context rows given to TabICL
N_TEST = 3000           # test rows scored
LAMBDAS = [8, 20]
TAUS = [0.80, 0.95]     # tail quantiles. MUST bracket the minority rate:
                        # a tail much larger than the minority upweights mostly
                        # hard MAJORITY points and hurts worst-group accuracy.

# Waterbirds groups: 0=landbird/land, 1=landbird/water, 2=waterbird/land, 3=waterbird/water
# Minority (spurious-correlation-breaking) groups are 1 and 2.
MINORITY_GROUPS = (1, 2)
GROUP_NAMES = ["landbird\non land", "landbird\non water",
               "waterbird\non land", "waterbird\non water"]

In [ ]:
# =====================================================================
# 1. Data: frozen features + labels + group labels (groups ONLY for eval)
# =====================================================================
def load_wilds_features():
    """Waterbirds -> frozen ResNet-50 penultimate features (the DFR setting).

    The `wilds` package's own download (CodaLab worksheets) is frequently
    down / 500s / returns 0 bytes -- this is a known server-side flakiness,
    not something wrong with your setup. This loader instead pulls the
    IDENTICAL archive directly from the Stanford DRO mirror that `wilds`
    itself wraps (same images, same metadata.csv schema:
    img_id,img_filename,y,split,place,place_filename; split 0/1/2 =
    train/val/test; group = 2*y + place), so results match `wilds` exactly.
    Falls back to `wilds.get_dataset` if the direct mirror is unreachable.

    NOTE ON PROTOCOL: canonical Waterbirds numbers (~97% mean / ~60% worst-group)
    come from features of a ResNet-50 *fine-tuned with ERM on Waterbirds*. This
    function uses ImageNet-pretrained features WITHOUT fine-tuning, which is
    cheaper but a different (usually weaker-spurious) feature space. The script
    prints the baseline per-group gap so you can see which regime you are in --
    if there is no gap, the experiment is uninformative, exactly as in the
    two-moons case. Set FINETUNE=True to reproduce the canonical setting.
    """
    import os, tarfile, urllib.request
    import torch, torchvision
    import pandas as pd
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset

    root = "data/waterbirds_direct"
    img_dir = os.path.join(root, "waterbird_complete95_forest2water2")
    meta_path = os.path.join(img_dir, "metadata.csv")

    if not os.path.exists(meta_path):
        os.makedirs(root, exist_ok=True)
        tar_path = os.path.join(root, "archive.tar.gz")
        mirrors = [
            "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz",
            "https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz",
        ]
        ok = False
        for url in mirrors:
            try:
                print(f"downloading {url} ...")
                urllib.request.urlretrieve(url, tar_path)
                ok = os.path.getsize(tar_path) > 10_000_000   # sanity: ~1.1GB expected
                if ok:
                    break
                print(f"  got {os.path.getsize(tar_path)} bytes, looks truncated, trying next mirror")
            except Exception as e:
                print(f"  failed: {type(e).__name__}: {e}")
        if not ok:
            print("direct mirrors failed too; falling back to wilds.get_dataset "
                  "(will hit the same CodaLab outage if that is the cause)")
            from wilds import get_dataset
            return _load_via_wilds(get_dataset(dataset="waterbirds", download=True))
        print("extracting ...")
        with tarfile.open(tar_path) as t:
            t.extractall(root)

    meta = pd.read_csv(meta_path)
    meta["group"] = 2 * meta["y"] + meta["place"]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tf = torchvision.transforms.Compose([
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225]),
    ])

    class WBImages(Dataset):
        def __init__(self, df):
            self.df = df.reset_index(drop=True)
        def __len__(self):
            return len(self.df)
        def __getitem__(self, i):
            row = self.df.iloc[i]
            img = Image.open(os.path.join(img_dir, row["img_filename"])).convert("RGB")
            return tf(img), int(row["y"]), int(row["group"])

    net = torchvision.models.resnet50(
        weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    net.fc = torch.nn.Identity()
    net = net.eval().to(device)

    def featurize(split_code):
        df = meta[meta["split"] == split_code]
        dl = DataLoader(WBImages(df), batch_size=128, shuffle=False, num_workers=2)
        F, Y, G = [], [], []
        with torch.no_grad():
            for x, y, g in dl:
                F.append(net(x.to(device)).cpu().numpy())
                Y.append(y.numpy()); G.append(g.numpy())
        return np.concatenate(F), np.concatenate(Y), np.concatenate(G)

    Xtr, ytr, gtr = featurize(0)     # train
    Xte, yte, gte = featurize(2)     # test
    return Xtr, ytr, gtr, Xte, yte, gte


def _load_via_wilds(ds):
    """Fallback path through the wilds package, if the direct mirror is
    also unreachable. Same feature extraction as load_wilds_features."""
    import torch, torchvision
    from torch.utils.data import DataLoader
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tf = torchvision.transforms.Compose([
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225]),
    ])
    net = torchvision.models.resnet50(
        weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    net.fc = torch.nn.Identity()
    net = net.eval().to(device)

    def featurize(split):
        sub = ds.get_subset(split, transform=tf)
        dl = DataLoader(sub, batch_size=128, shuffle=False, num_workers=2)
        F, Y, G = [], [], []
        with torch.no_grad():
            for x, y, meta in dl:
                F.append(net(x.to(device)).cpu().numpy())
                Y.append(y.numpy())
                bg = meta[:, 0].numpy()
                G.append(2 * y.numpy() + bg)
        return np.concatenate(F), np.concatenate(Y), np.concatenate(G)

    Xtr, ytr, gtr = featurize("train")
    Xte, yte, gte = featurize("test")
    return Xtr, ytr, gtr, Xte, yte, gte


def load_synthetic():
    """Waterbirds-SHAPED sanity data: core feature (weak, always valid) +
    spurious feature (strong, valid for 95% majority, flipped for minority)
    + noise dims. Same 4-group structure, so all bookkeeping is exercised."""
    rng = np.random.default_rng(SEED)

    def make(n, spur_corr):
        y = rng.integers(0, 2, n)                       # 0=landbird, 1=waterbird
        agree = rng.random(n) < spur_corr
        bg = np.where(agree, y, 1 - y)                  # background
        g = 2 * y + bg
        core = 0.8 * (2 * y - 1) + 1.0 * rng.standard_normal(n)
        spur = 2.5 * (2 * bg - 1) + 0.4 * rng.standard_normal(n)
        noise = rng.standard_normal((n, 30))
        return np.column_stack([core, spur, noise]), y, g

    Xtr, ytr, gtr = make(4000, 0.95)     # train: strong spurious correlation
    Xte, yte, gte = make(3000, 0.50)     # test: correlation broken
    return Xtr, ytr, gtr, Xte, yte, gte

In [ ]:
# =====================================================================
# 2. Backend
# =====================================================================
def build_clf():
    if BACKEND == "tabicl":
        import torch
        from tabicl import TabICLClassifier
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        return TabICLClassifier(n_estimators=1, device=dev, random_state=SEED,
                                offload_mode=False)
    from sklearn.linear_model import LogisticRegression
    return LogisticRegression(max_iter=2000)   # a "last layer", like DFR's

def fit_predict(Xc, yc, Xq):
    clf = build_clf()
    clf.fit(np.asarray(Xc, np.float32), np.asarray(yc))
    return clf.predict(np.asarray(Xq, np.float32)), clf

In [ ]:
# =====================================================================
# 3. Difficulty score
# =====================================================================
def gaussian_kernel(A, B, gamma):
    a2 = np.sum(A * A, 1)[:, None]; b2 = np.sum(B * B, 1)[None, :]
    return np.exp(-gamma * np.maximum(a2 + b2 - 2 * A @ B.T, 0.0))

def incontext_loo_nll(K, y, eps=1e-12):
    K = K.copy(); np.fill_diagonal(K, 0.0)
    same = (y[:, None] == y[None, :]).astype(float)
    return -np.log(np.clip((K * same).sum(1) / (K.sum(1) + eps), eps, 1.0))

def difficulty(Xc, yc):
    """Exact embedding-space LOO when the TabICL hook is available;
    cross-fitted out-of-fold NLL otherwise. Returns (L, source_str)."""
    if BACKEND == "tabicl":
        try:
            import torch
            clf = build_clf(); clf.fit(np.asarray(Xc, np.float32), np.asarray(yc))
            target = clf.model_.icl_predictor.tf_icl
            blocks = []
            def hook(_m, _i, out):
                t = out[0] if isinstance(out, (tuple, list)) else out
                blocks.append(t.detach().float().cpu())
            h = target.register_forward_hook(hook)
            try:
                clf.predict_proba(np.asarray(Xc, np.float32))
            finally:
                h.remove()
            E = torch.cat([b[0, len(Xc):] for b in blocks], 0).numpy()
            assert E.shape[0] == len(Xc), (E.shape, len(Xc))
            g = 1.0 / (2.0 * np.sqrt(E.shape[1]))
            return incontext_loo_nll(gaussian_kernel(E, E, g), np.asarray(yc)), "embedding"
        except Exception as e:
            print(f"[warn] embedding hook failed ({type(e).__name__}: {e}); "
                  f"falling back to cross-fit")
    # cross-fitted fallback (works for any black-box head)
    Xc = np.asarray(Xc, np.float32); yc = np.asarray(yc)
    skf = StratifiedKFold(4, shuffle=True, random_state=SEED)
    pt = np.zeros(len(yc))
    for tr, oof in skf.split(Xc, yc):
        clf = build_clf(); clf.fit(Xc[tr], yc[tr])
        p1 = clf.predict_proba(Xc[oof])[:, 1]
        pt[oof] = np.where(yc[oof] == 1, p1, 1 - p1)
    return -np.log(np.clip(pt, 1e-12, 1.0)), "crossfit"

In [ ]:
# =====================================================================
# 4. Metrics + context construction
# =====================================================================
def group_metrics(pred, y, g):
    per = {gi: float(np.mean(pred[g == gi] == y[g == gi]))
           for gi in range(4) if (g == gi).any()}
    return dict(mean=float(np.mean(pred == y)), per_group=per,
                worst=min(per.values()))

def resample(X, y, w, rng):
    p = w / w.sum()
    idx = rng.choice(len(y), size=len(y), replace=True, p=p)
    return X[idx], y[idx]

def run(seed, Xtr, ytr, gtr, Xte, yte, gte, L):
    rng = np.random.default_rng(seed)
    out = {}
    pred, _ = fit_predict(Xtr, ytr, Xte)
    out["baseline"] = group_metrics(pred, yte, gte)

    # Tail-set upweighting. TAU MATTERS MORE THAN LAMBDA: the tail must be
    # about the size of the minority, or most of the upweight lands on hard
    # MAJORITY points and worst-group accuracy gets worse than doing nothing.
    for tau in TAUS:
        thr = np.quantile(L, tau)
        for lam in LAMBDAS:
            if lam == 0:
                continue
            w = 1.0 + lam * (L >= thr)
            Xc, yc = resample(Xtr, ytr, w, rng)
            out[f"tail_t{tau}_l{lam}"] = group_metrics(
                fit_predict(Xc, yc, Xte)[0], yte, gte)

    # Group-free DFR analogue: pseudo-groups = (class) x (hard/easy by LOO),
    # then BALANCE the four cells -- mirrors DFR's group balancing without
    # ever using a group label.
    for tau in TAUS:
        hard = L >= np.quantile(L, tau)
        w = np.ones(len(ytr))
        for yy in np.unique(ytr):
            for hh in (False, True):
                m = (ytr == yy) & (hard == hh)
                if m.sum() > 0:
                    w[m] = len(ytr) / (4.0 * m.sum())
        Xc, yc = resample(Xtr, ytr, w, rng)
        out[f"pseudobal_t{tau}"] = group_metrics(
            fit_predict(Xc, yc, Xte)[0], yte, gte)

    # control: reweight a random tail of the same size
    w = np.ones(len(ytr)); w[rng.choice(len(ytr), int(0.05 * len(ytr)),
                                        replace=False)] += 20.0
    Xc, yc = resample(Xtr, ytr, w, rng)
    out["random_ctrl"] = group_metrics(fit_predict(Xc, yc, Xte)[0], yte, gte)

    # DFR-style oracle: group-balanced context (USES GROUP LABELS)
    w = np.ones(len(ytr))
    for gi in range(4):
        m = gtr == gi
        if m.sum() > 0:
            w[m] = len(ytr) / (4.0 * m.sum())
    Xc, yc = resample(Xtr, ytr, w, rng)
    out["dfr_oracle"] = group_metrics(fit_predict(Xc, yc, Xte)[0], yte, gte)
    return out


In [ ]:
# =====================================================================
# 5. Main
# =====================================================================
def main(n_seeds=3):
    if DATA == "wilds":
        Xtr, ytr, gtr, Xte, yte, gte = load_wilds_features()
    else:
        Xtr, ytr, gtr, Xte, yte, gte = load_synthetic()

    rng = np.random.default_rng(SEED)
    itr = rng.choice(len(ytr), min(N_CONTEXT, len(ytr)), replace=False)
    ite = rng.choice(len(yte), min(N_TEST, len(yte)), replace=False)
    Xtr, ytr, gtr = Xtr[itr], ytr[itr], gtr[itr]
    Xte, yte, gte = Xte[ite], yte[ite], gte[ite]

    if Xtr.shape[1] > N_PCA:
        pca = PCA(N_PCA, random_state=SEED).fit(Xtr)
        Xtr, Xte = pca.transform(Xtr), pca.transform(Xte)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

    print(f"context {Xtr.shape}, test {Xte.shape}")
    print("train group sizes:", {gi: int((gtr == gi).sum()) for gi in range(4)})

    L, src = difficulty(Xtr, ytr)

    # ---- THE DECISIVE DIAGNOSTIC (report before any accuracy) ----------
    is_min = np.isin(gtr, MINORITY_GROUPS)
    auc = roc_auc_score(is_min, L)
    print("\n" + "=" * 62)
    print(f"DIAGNOSTIC  minority-detection AUC = {auc:.3f}   (score: {src})")
    print("  0.5 = LOO carries no information about group membership.")
    print("  If this is ~0.5, the reweighting below CANNOT work; report this")
    print("  as the finding rather than tuning lambda until something moves.")
    print("=" * 62 + "\n")

    runs = [run(s, Xtr, ytr, gtr, Xte, yte, gte, L) for s in range(n_seeds)]
    conds = list(runs[0].keys())
    agg = {c: dict(mean=np.mean([r[c]["mean"] for r in runs]),
                   worst=np.mean([r[c]["worst"] for r in runs]),
                   worst_sd=np.std([r[c]["worst"] for r in runs]))
           for c in conds}

    print(f"{'condition':>14} {'mean':>7} {'worst':>7} {'sd':>6}")
    for c in conds:
        print(f"{c:>14} {agg[c]['mean']:>7.3f} {agg[c]['worst']:>7.3f} "
              f"{agg[c]['worst_sd']:>6.3f}")
    n_min = int(np.isin(gte, MINORITY_GROUPS).sum())
    print(f"\n[stats] {n_min} minority test points -> SE on a worst-group "
          f"proportion near 0.7 is ~{np.sqrt(0.7*0.3/max(n_min,1)):.3f}. "
          f"Treat differences below ~2 SE as noise.")

    # ---- figures -------------------------------------------------------
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))

    # (1) LOO score distribution by group: does the score see the minority?
    ax[0].boxplot([L[gtr == gi] for gi in range(4)], labels=GROUP_NAMES,
                  showfliers=False)
    ax[0].set_ylabel("LOO difficulty $L_i$")
    ax[0].set_title(f"(1) Does LOO see the minority?  AUC={auc:.2f}")
    for i, gi in enumerate(range(4)):
        if gi in MINORITY_GROUPS:
            ax[0].get_xticklabels()[i].set_color("crimson")

    # (2) worst-group accuracy by condition, with the oracle as reference
    cs = [c for c in conds]
    ax[1].bar(range(len(cs)), [agg[c]["worst"] for c in cs],
              yerr=[agg[c]["worst_sd"] for c in cs], capsize=3,
              color=["gray" if c == "baseline" else
                     "tab:orange" if c == "dfr_oracle" else
                     "lightsteelblue" if c == "random_ctrl" else "tab:blue"
                     for c in cs])
    ax[1].axhline(agg["baseline"]["worst"], ls=":", c="k", lw=1)
    ax[1].set_xticks(range(len(cs)))
    ax[1].set_xticklabels(cs, rotation=30, ha="right", fontsize=8)
    ax[1].set_ylabel("worst-group accuracy")
    ax[1].set_title("(2) Worst-group vs DFR-style oracle")

    # (3) per-group accuracy, baseline vs best ours vs oracle
    best = max([c for c in conds if c.startswith(("tail", "pseudobal"))],
               key=lambda c: agg[c]["worst"])
    w = 0.26
    for k, (c, lbl) in enumerate([("baseline", "baseline"), (best, best),
                                  ("dfr_oracle", "DFR oracle")]):
        vals = [np.mean([r[c]["per_group"].get(gi, np.nan) for r in runs])
                for gi in range(4)]
        ax[2].bar(np.arange(4) + (k - 1) * w, vals, w, label=lbl)
    ax[2].set_xticks(range(4)); ax[2].set_xticklabels(GROUP_NAMES, fontsize=8)
    ax[2].set_ylabel("accuracy"); ax[2].set_ylim(0, 1)
    ax[2].legend(fontsize=8); ax[2].set_title("(3) Per-group accuracy")

    fig.suptitle(f"Waterbirds ({DATA}/{BACKEND}): group-free LOO reweighting "
                 f"vs group-labeled DFR", fontsize=13)
    fig.tight_layout()
    fig.savefig("waterbirds_results.png", dpi=140, bbox_inches="tight")
    plt.show()
    print("saved waterbirds_results.png")

if __name__ == "__main__":
    main()

downloading https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz ...
extracting ...


/tmp/ipykernel_2693/3289879244.py:59: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(root)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 152MB/s]


context (4000, 128), test (3000, 128)
train group sizes: {0: 2920, 1: 149, 2: 42, 3: 889}


...

tabicl-classifier-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  110MB            

tabicl-classifier-v2-20260212.ckpt: downloading bytes:           |  0.00B            


DIAGNOSTIC  minority-detection AUC = 0.830   (score: embedding)
  0.5 = LOO carries no information about group membership.
  If this is ~0.5, the reweighting below CANNOT work; report this
  as the finding rather than tuning lambda until something moves.

     condition    mean   worst     sd
      baseline   0.858   0.657  0.000
  tail_t0.8_l8   0.841   0.602  0.013
 tail_t0.8_l20   0.830   0.568  0.007
 tail_t0.95_l8   0.834   0.603  0.016
tail_t0.95_l20   0.829   0.523  0.034
pseudobal_t0.8   0.855   0.626  0.007
pseudobal_t0.95   0.813   0.620  0.017
   random_ctrl   0.847   0.594  0.011
    dfr_oracle   0.843   0.624  0.005

[stats] 1485 minority test points -> SE on a worst-group proportion near 0.7 is ~0.012. Treat differences below ~2 SE as noise.


/tmp/ipykernel_2693/3320187661.py:57: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax[0].boxplot([L[gtr == gi] for gi in range(4)], labels=GROUP_NAMES,


saved waterbirds_results.png
